
# Boundary-Conditioned Spectral Analysis

Proves that learned boundaries correspond to genuine neurophysiological
changes, not just learned artifacts.

Method:
   1. Run trained model on test data, extract boundary activations
   2. Split tokens into HIGH boundary (top 20%) vs LOW boundary (bottom 20%)
   3. Map tokens back to raw EEG time windows
   4. Compute spectral features (band powers, spectral entropy) for each group
   5. Statistical comparison: do high-boundary regions have different
      frequency content than low-boundary regions?

Expected results:
   Sleep-EDF: high-boundary regions should show frequency transitions
              (e.g., delta↔alpha shifts between stages)
   CHB-MIT: high-boundary regions should show spectral changes
            characteristic of seizure onset (increased beta/gamma)

Applies to: Sleep-EDF and CHB-MIT (not TUAB — boundary std too low)


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import h5py
import json
from scipy.signal import welch
from scipy.stats import mannwhitneyu, ttest_ind
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

figures_dir = Path("figures/spectral")
figures_dir.mkdir(parents=True, exist_ok=True)

Device: cuda


In [2]:
# MODEL 

class PatchEmbedding(nn.Module):
    def __init__(self,nc=3,ns=3000,ed=128,tk=25,pk=75,ps=15,do=0.1):
        super().__init__()
        self.tc=nn.Sequential(nn.Conv2d(1,40,(1,tk),padding=(0,tk//2)),nn.BatchNorm2d(40),nn.GELU())
        self.sc=nn.Sequential(nn.Conv2d(40,40,(nc,1)),nn.BatchNorm2d(40),nn.GELU())
        self.pool=nn.AvgPool2d((1,pk),stride=(1,ps))
        self.proj=nn.Sequential(nn.Conv2d(40,ed,(1,1)),nn.Dropout(do))
        self.seq_len=(ns-pk)//ps+1
    def forward(self,x):
        x=x.unsqueeze(1);x=self.tc(x);x=self.sc(x);x=self.pool(x);x=self.proj(x);return x.squeeze(2).permute(0,2,1)

class MultiResolutionEncoder(nn.Module):
    def __init__(self,nc=3,ns=3000,ed=128,do=0.1):
        super().__init__()
        self.enc_100=PatchEmbedding(nc,ns,ed,do=do);self.enc_50=PatchEmbedding(nc,ns//2,ed,do=do);self.enc_25=PatchEmbedding(nc,ns//4,ed,do=do)
        self.merge=nn.Sequential(nn.Linear(ed*3,ed),nn.GELU(),nn.Dropout(do));self.seq_len_100=self.enc_100.seq_len
    def forward(self,x):
        e1=self.enc_100(x);e2=self.enc_50(x[:,:,::2]);e3=self.enc_25(x[:,:,::4]);T=e1.shape[1]
        e2=F.interpolate(e2.permute(0,2,1),size=T,mode='linear',align_corners=False).permute(0,2,1)
        e3=F.interpolate(e3.permute(0,2,1),size=T,mode='linear',align_corners=False).permute(0,2,1)
        return self.merge(torch.cat([e1,e2,e3],dim=-1))

class ContrastiveBoundaryModule(nn.Module):
    def __init__(self,ed=128,hd=64,scales=(1,4,16),do=0.1):
        super().__init__()
        self.scales=scales
        self.projections=nn.ModuleList([nn.Sequential(nn.Linear(ed,hd),nn.GELU(),nn.Dropout(do),nn.Linear(hd,hd)) for _ in scales])
        self.fusion=nn.Sequential(nn.Linear(len(scales),len(scales)*2),nn.GELU(),nn.Linear(len(scales)*2,1))
        self.temperature=nn.Parameter(torch.tensor(1.0))
    def _contrast(self,x,proj,offset):
        B,T,D=x.shape;h=F.normalize(proj(x),dim=-1)
        if offset<T:
            hs=torch.roll(h,-offset,dims=1);hs[:,-offset:,:]=h[:,-offset:,:]
            c=1.0-((h*hs).sum(dim=-1)+1.0)/2.0;c=torch.sigmoid((c-0.5)*self.temperature.abs().clamp(min=0.1))
        else:c=torch.zeros(B,T,device=x.device)
        return c
    def forward(self,x):
        ps=[self._contrast(x,p,o) for p,o in zip(self.projections,self.scales)]
        f=torch.sigmoid(self.fusion(torch.stack(ps,dim=-1)).squeeze(-1))
        return {'boundaries':f,'boundary_loss':0.01*sum(F.mse_loss(p,f.detach()) for p in ps)/len(ps)}

class OriginalRegimeMask(nn.Module):
    def __init__(self):super().__init__()
    def forward(self,b):c=torch.cumsum(b,dim=1);s=torch.exp(-torch.abs(c.unsqueeze(2)-c.unsqueeze(1)));return s,1-s

class RegimeStructuredAttention(nn.Module):
    def __init__(self,ed=128,ni=4,nit=2,nc=2,do=0.1,rmm=None):
        super().__init__()
        self.n_heads=ni+nit+nc;self.head_dim=ed//self.n_heads;self.n_intra=ni;self.n_inter=nit;self.scale=self.head_dim**-0.5
        self.qkv=nn.Linear(ed,3*ed);self.out_proj=nn.Linear(ed,ed);self.attn_drop=nn.Dropout(do);self.proj_drop=nn.Dropout(do);self.regime_mask=rmm or OriginalRegimeMask()
    def forward(self,x,boundaries,return_attention=False):
        B,T,D=x.shape;qkv=self.qkv(x).reshape(B,T,3,self.n_heads,self.head_dim).permute(2,0,3,1,4);q,k,v=qkv[0],qkv[1],qkv[2]
        attn=(q@k.transpose(-2,-1))*self.scale;s,c=self.regime_mask(boundaries);s=s.unsqueeze(1);c=c.unsqueeze(1)
        mask=torch.ones_like(attn);mask[:,:self.n_intra]=s.expand(B,self.n_intra,T,T);mask[:,self.n_intra:self.n_intra+self.n_inter]=c.expand(B,self.n_inter,T,T)
        attn=attn+torch.log(mask+1e-6);w=F.softmax(attn,dim=-1);w=self.attn_drop(w);out=(w@v).transpose(1,2).reshape(B,T,D);out=self.proj_drop(self.out_proj(out))
        if return_attention:return out,w
        return out

class NeuroStateBlock(nn.Module):
    def __init__(self,ed=128,ni=4,nit=2,nc=2,mr=4.0,do=0.1,rmm=None):
        super().__init__()
        self.norm1=nn.LayerNorm(ed);self.attn=RegimeStructuredAttention(ed,ni,nit,nc,do,rmm)
        self.norm2=nn.LayerNorm(ed);self.mlp=nn.Sequential(nn.Linear(ed,int(ed*mr)),nn.GELU(),nn.Dropout(do),nn.Linear(int(ed*mr),ed),nn.Dropout(do))
    def forward(self,x,b,ra=False):
        if ra:a,w=self.attn(self.norm1(x),b,True);x=x+a;x=x+self.mlp(self.norm2(x));return x,w
        x=x+self.attn(self.norm1(x),b);x=x+self.mlp(self.norm2(x));return x

class MultiResContrastiveNeuroState(nn.Module):
    def __init__(self,nc=3,ns=3000,ncls=5,ed=128,nl=4,do=0.1,cs=(1,4,16),ch=64,nce=3):
        super().__init__()
        self.n_classes=ncls;self.n_context=nce
        self.mr_encoder=MultiResolutionEncoder(nc,ns,ed,do)
        self.tokens_per_epoch=self.mr_encoder.seq_len_100;tt=self.tokens_per_epoch*nce
        self.pos_embed=nn.Parameter(torch.randn(1,tt,ed)*0.02);self.pos_drop=nn.Dropout(do)
        self.epoch_embed=nn.Parameter(torch.randn(1,nce,1,ed)*0.02)
        self.changepoint_module=ContrastiveBoundaryModule(ed,ch,cs,do)
        self.blocks=nn.ModuleList([NeuroStateBlock(ed,do=do) for _ in range(nl)])
        self.norm=nn.LayerNorm(ed)
        self.head=nn.Sequential(nn.Linear(ed,ed//2),nn.GELU(),nn.Dropout(do),nn.Linear(ed//2,ncls))
        self.apply(self._iw)
    def _iw(self,m):
        if isinstance(m,nn.Linear):nn.init.trunc_normal_(m.weight,std=0.02)
        if hasattr(m,'bias') and m.bias is not None:nn.init.zeros_(m.bias)

In [3]:
# Extract boundary activations and map to raw EEG

def extract_boundary_spectral_pairs(model, epochs, labels, subject_ids,
                                     n_channels, sfreq, device,
                                     max_windows=500):
    """
    For each 3-epoch window, extract boundary activation per token
    and compute spectral features for the corresponding raw EEG segment.

    Returns list of dicts with per-token boundary value and spectral features.
    """
    model.eval()
    tpe = model.tokens_per_epoch
    samples_per_token = int(sfreq * 30.0 / tpe)  # ~15 samples per token
    results = []

    # Find valid windows
    unique_subj = np.unique(subject_ids)
    window_count = 0

    for subj in unique_subj:
        subj_mask = subject_ids == subj
        subj_idx = np.where(subj_mask)[0]

        for i in range(1, len(subj_idx) - 1):
            center = subj_idx[i]
            prev, nxt = subj_idx[i-1], subj_idx[i+1]
            if nxt - prev != 2:
                continue

            # Build 3-epoch window
            eps = []
            for idx in [prev, center, nxt]:
                ep = epochs[idx].astype(np.float32)
                nc = ep.shape[0]
                if nc < n_channels:
                    ep = np.vstack([ep, np.zeros((n_channels-nc, ep.shape[1]), dtype=np.float32)])
                elif nc > n_channels:
                    ep = ep[:n_channels]
                eps.append(ep)

            x = torch.tensor(np.stack(eps), dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                B, N, C, T = x.shape
                embs = [model.mr_encoder(x[:,j]) + model.epoch_embed[:,j] for j in range(N)]
                full = model.pos_drop(torch.cat(embs, dim=1) + model.pos_embed)
                cp = model.changepoint_module(full)
                bnd = cp['boundaries'][0].cpu().numpy()  # (588,)

            # For center epoch only (tokens tpe to 2*tpe)
            center_bnd = bnd[tpe:2*tpe]  # (196,)
            center_eeg = epochs[center]   # (C, 3000)

            # Compute spectral features for each token's time window
            for t_idx in range(len(center_bnd)):
                sample_start = t_idx * samples_per_token
                sample_end = min(sample_start + samples_per_token * 2, center_eeg.shape[1])
                if sample_end - sample_start < 16:
                    continue

                segment = center_eeg[:n_channels, sample_start:sample_end]

                # Band powers
                bands = {'delta':(0.5,4),'theta':(4,8),'alpha':(8,13),
                         'sigma':(11,16),'beta':(13,30),'gamma':(30,45)}
                powers = {}
                for ch in range(min(n_channels, segment.shape[0])):
                    try:
                        freqs, psd = welch(segment[ch], fs=sfreq,
                                          nperseg=min(64, segment.shape[1]))
                        for bn, (lo, hi) in bands.items():
                            m = (freqs >= lo) & (freqs <= hi)
                            bp = np.trapz(psd[m], freqs[m]) if m.any() else 0
                            if bn not in powers:
                                powers[bn] = []
                            powers[bn].append(bp)
                    except Exception:
                        pass

                if not powers:
                    continue

                avg_powers = {k: np.mean(v) for k, v in powers.items()}

                # Spectral entropy
                total = sum(avg_powers.values()) + 1e-10
                probs = np.array([v/total for v in avg_powers.values()])
                spectral_entropy = -np.sum(probs * np.log(probs + 1e-10))

                results.append({
                    'boundary_val': float(center_bnd[t_idx]),
                    'label': int(labels[center]),
                    **avg_powers,
                    'spectral_entropy': spectral_entropy,
                    'amplitude_std': float(np.std(segment)),
                })

            window_count += 1
            if window_count >= max_windows:
                break
        if window_count >= max_windows:
            break

    print(f"  Extracted {len(results)} token-level spectral pairs "
          f"from {window_count} windows")
    return results

In [4]:
# Compare high vs low boundary regions

def analyze_boundary_spectral(results, dataset_name, save_dir):
    """
    Split into high/low boundary groups and compare spectral features.
    """
    bnd_vals = np.array([r['boundary_val'] for r in results])
    high_thresh = np.percentile(bnd_vals, 80)
    low_thresh = np.percentile(bnd_vals, 20)

    high_mask = bnd_vals >= high_thresh
    low_mask = bnd_vals <= low_thresh

    print(f"\n--- {dataset_name}: High vs Low Boundary Spectral Comparison ---")
    print(f"  High boundary (>{high_thresh:.4f}): {high_mask.sum()} tokens")
    print(f"  Low boundary (<{low_thresh:.4f}): {low_mask.sum()} tokens")

    features = ['delta', 'theta', 'alpha', 'sigma', 'beta', 'gamma',
                'spectral_entropy', 'amplitude_std']
    comparison = {}

    for feat in features:
        vals = np.array([r.get(feat, 0) for r in results])
        high_vals = vals[high_mask]
        low_vals = vals[low_mask]

        # Statistical test
        try:
            stat, pval = mannwhitneyu(high_vals, low_vals, alternative='two-sided')
        except Exception:
            stat, pval = 0, 1.0

        ratio = np.mean(high_vals) / (np.mean(low_vals) + 1e-10)
        comparison[feat] = {
            'high_mean': float(np.mean(high_vals)),
            'low_mean': float(np.mean(low_vals)),
            'ratio': float(ratio),
            'p_value': float(pval),
            'significant': bool(pval < 0.05),
        }

        sig_marker = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"  {feat:>18s}: high={np.mean(high_vals):.4f} low={np.mean(low_vals):.4f} "
              f"ratio={ratio:.3f} p={pval:.4f} {sig_marker}")

    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # Band power comparison
    ax = axes[0, 0]
    band_features = ['delta', 'theta', 'alpha', 'sigma', 'beta', 'gamma']
    x = np.arange(len(band_features))
    high_means = [comparison[f]['high_mean'] for f in band_features]
    low_means = [comparison[f]['low_mean'] for f in band_features]
    w = 0.35
    ax.bar(x - w/2, high_means, w, label='High boundary', color='#A32638', alpha=0.8)
    ax.bar(x + w/2, low_means, w, label='Low boundary', color='#3498db', alpha=0.8)
    # Add significance stars
    for i, f in enumerate(band_features):
        if comparison[f]['significant']:
            max_val = max(high_means[i], low_means[i])
            ax.text(i, max_val * 1.05, '*', ha='center', fontsize=14, color='red')
    ax.set_xticks(x)
    ax.set_xticklabels(band_features, fontsize=10)
    ax.set_ylabel('Mean band power', fontsize=10)
    ax.set_title('Band Powers: High vs Low Boundary', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

    # Spectral entropy
    ax = axes[0, 1]
    se_high = [r['spectral_entropy'] for r in results if r['boundary_val'] >= high_thresh]
    se_low = [r['spectral_entropy'] for r in results if r['boundary_val'] <= low_thresh]
    ax.hist(se_low, bins=30, alpha=0.6, label='Low boundary', color='#3498db')
    ax.hist(se_high, bins=30, alpha=0.6, label='High boundary', color='#A32638')
    ax.set_xlabel('Spectral entropy', fontsize=10)
    ax.set_title('Spectral Entropy Distribution', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

    # Amplitude std
    ax = axes[1, 0]
    amp_high = [r['amplitude_std'] for r in results if r['boundary_val'] >= high_thresh]
    amp_low = [r['amplitude_std'] for r in results if r['boundary_val'] <= low_thresh]
    ax.hist(amp_low, bins=30, alpha=0.6, label='Low boundary', color='#3498db')
    ax.hist(amp_high, bins=30, alpha=0.6, label='High boundary', color='#A32638')
    ax.set_xlabel('Amplitude std', fontsize=10)
    ax.set_title('Amplitude Variability', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

    # Boundary value distribution
    ax = axes[1, 1]
    ax.hist(bnd_vals, bins=50, color='#555', alpha=0.7)
    ax.axvline(high_thresh, color='#A32638', linestyle='--', linewidth=2,
               label=f'High thresh ({high_thresh:.3f})')
    ax.axvline(low_thresh, color='#3498db', linestyle='--', linewidth=2,
               label=f'Low thresh ({low_thresh:.3f})')
    ax.set_xlabel('Boundary activation', fontsize=10)
    ax.set_title('Boundary Value Distribution', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)

    plt.suptitle(f'Boundary-Conditioned Spectral Analysis — {dataset_name}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    save_path = save_dir / f'spectral_analysis_{dataset_name.lower().replace("-","")}.png'
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")

    return comparison

In [5]:
# MAIN

def main():
    print("=" * 60)
    print("BOUNDARY-CONDITIONED SPECTRAL ANALYSIS")
    print("=" * 60)

    # --- Sleep-EDF ---
    sleep_h5 = Path("data/processed/sleep_edf_processed.h5")
    sleep_ckpt = Path("models/neurostate_acbl_isolated.pt")

    if sleep_h5.exists() and sleep_ckpt.exists():
        print("\n--- Sleep-EDF ---")
        model = MultiResContrastiveNeuroState(nc=3, ns=3000, ncls=5).to(device)
        ckpt = torch.load(sleep_ckpt, map_location=device, weights_only=False)
        state = ckpt['model_state_dict']
        fixed = {}
        for k, v in state.items():
            k = k.replace('enc_100hz', 'enc_100').replace('enc_50hz', 'enc_50').replace('enc_25hz', 'enc_25')
            k = k.replace('temporal_conv', 'tc').replace('spatial_conv', 'sc').replace('projection', 'proj')
            fixed[k] = v
        model.load_state_dict(fixed, strict=False)
        print(f"  Loaded: {sleep_ckpt}")

        with h5py.File(sleep_h5, 'r') as f:
            epochs = f['epochs'][:]
            labels = f['labels'][:]
            subject_ids = np.array([s.decode() if isinstance(s, bytes) else str(s)
                                    for s in f['subject_ids'][:]])

        print("\nExtracting boundary-spectral pairs:")
        results_sleep = extract_boundary_spectral_pairs(
            model, epochs, labels, subject_ids,
            n_channels=3, sfreq=100, device=device, max_windows=300)

        comparison_sleep = analyze_boundary_spectral(
            results_sleep, 'Sleep-EDF', figures_dir)

        with open(figures_dir / 'spectral_sleep.json', 'w') as f:
            json.dump(comparison_sleep, f, indent=2)

    # --- CHB-MIT ---
    chb_h5 = Path("data/processed/chbmit_combined_seizure_detection.h5")
    chb_ckpt = Path("models/acbl_isolation_chbmit_best.pt")

    if chb_h5.exists() and chb_ckpt.exists():
        print("\n--- CHB-MIT ---")
        model = MultiResContrastiveNeuroState(nc=17, ns=3000, ncls=2).to(device)
        ckpt = torch.load(chb_ckpt, map_location=device, weights_only=False)
        state = ckpt['model_state_dict']
        fixed = {}
        for k, v in state.items():
            k = k.replace('temporal_conv', 'tc').replace('spatial_conv', 'sc').replace('projection', 'proj')
            fixed[k] = v
        model.load_state_dict(fixed, strict=False)
        print(f"  Loaded: {chb_ckpt}")

        with h5py.File(chb_h5, 'r') as f:
            epochs = f['epochs'][:]
            labels = f['labels'][:]
            subject_ids = np.array([s.decode() for s in f['subject_ids'][:]])

        print("\nExtracting boundary-spectral pairs:")
        results_chb = extract_boundary_spectral_pairs(
            model, epochs, labels, subject_ids,
            n_channels=17, sfreq=100, device=device, max_windows=300)

        comparison_chb = analyze_boundary_spectral(
            results_chb, 'CHB-MIT', figures_dir)

        with open(figures_dir / 'spectral_chbmit.json', 'w') as f:
            json.dump(comparison_chb, f, indent=2)

    print("\nDone.")


if __name__ == "__main__":
    main()

BOUNDARY-CONDITIONED SPECTRAL ANALYSIS

--- Sleep-EDF ---
  Loaded: models/neurostate_acbl_isolated.pt

Extracting boundary-spectral pairs:
  Extracted 58800 token-level spectral pairs from 300 windows

--- Sleep-EDF: High vs Low Boundary Spectral Comparison ---
  High boundary (>0.2324): 11760 tokens
  Low boundary (<0.2096): 11760 tokens
               delta: high=0.0000 low=0.0000 ratio=0.000 p=1.0000 
               theta: high=0.0000 low=0.0000 ratio=0.000 p=1.0000 
               alpha: high=0.0000 low=0.0000 ratio=0.000 p=1.0000 
               sigma: high=0.0000 low=0.0000 ratio=0.000 p=1.0000 
                beta: high=0.0351 low=0.0167 ratio=2.105 p=0.0000 ***
               gamma: high=0.0136 low=0.0042 ratio=3.267 p=0.0000 ***
    spectral_entropy: high=0.4488 low=0.4338 ratio=1.035 p=0.0000 ***
       amplitude_std: high=0.8533 low=0.8392 ratio=1.017 p=0.4033 
  Saved: figures/spectral/spectral_analysis_sleepedf.png

--- CHB-MIT ---
  Loaded: models/acbl_isolation_chbmit_

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=30d601c2-0a51-44b0-ac6c-72cf48d1679e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>